In [12]:
# ============================================================
# TUNDA BOX JUNIOR DATA ANALYST ASSESSMENT
# SETUP: DATA CREATION AND DATABASE CONNECTION
# ============================================================

import sqlite3
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. CUSTOMER DATA
# ------------------------------------------------------------

customers = pd.DataFrame([
    [101, "Nairobi"],
    [102, "nairobi"],
    [103, "Mombasa"],
    [104, "Kisumu"],
    [105, "Nairobi "],
    [106, "Nakuru"],
    [107, "Mombasa"],
    [108, "Nairobi"],
    [109, "Eldoret"],
    [110, "Kisumu"]
], columns=["customer_id", "city"])


# ------------------------------------------------------------
# 2. ORDER DATA
# Raw data is intentionally kept as-is.
# This includes the duplicate, missing amount and invalid
# customer reference required for the assessment.
# ------------------------------------------------------------

orders = pd.DataFrame([
    [1001, 101, "2026-06-03", "completed", 2400],
    [1002, 102, "2026-06-05", "completed", 1850],
    [1003, 103, "2026-06-11", "cancelled", 3200],
    [1004, 104, "2026-06-18", "completed", 2100],
    [1005, 101, "2026-06-25", "completed", 2650],
    [1006, 105, "2026-07-02", "completed", 1500],
    [1006, 105, "2026-07-02", "completed", 1500],
    [1007, 106, "2026-07-09", "refunded", 2900],
    [1008, 102, "2026-07-14", "completed", 3100],
    [1009, 107, "2026-07-21", "completed", None],
    [1010, 101, "2026-07-28", "completed", 2200],
    [1011, 108, "2026-08-04", "completed", 1750],
    [1012, 103, "2026-08-09", "completed", 2800],
    [1013, 104, "2026-08-15", "completed", 185000],
    [1014, 105, "2026-08-19", "cancelled", 1600],
    [1015, 102, "2026-08-27", "completed", 2450],
    [1016, 106, "2026-09-03", "completed", 3300],
    [1017, 101, "2026-09-08", "completed", 2750],
    [1018, 108, "2026-09-14", "refunded", 2000],
    [1019, 107, "2026-09-22", "completed", 2350],
    [1020, 999, "2026-09-26", "completed", 1900]
], columns=[
    "order_id",
    "customer_id",
    "order_date",
    "status",
    "amount_kes"
])


# ------------------------------------------------------------
# 3. A/B TEST DATA
# ------------------------------------------------------------

ab = pd.DataFrame([
    ["Current", 1000, 50],
    ["New", 1000, 60]
], columns=["version", "visitors", "conversions"])


# ------------------------------------------------------------
# 4. MARKETING CHANNEL DATA
# ------------------------------------------------------------

channels = pd.DataFrame([
    ["September", "Organic", 40000, 2400],
    ["September", "Paid Social", 10000, 200],
    ["October", "Organic", 38000, 2318],
    ["October", "Paid Social", 30000, 630]
], columns=[
    "month",
    "channel",
    "visitors",
    "conversions"
])


# ------------------------------------------------------------
# 5. CREATE IN-MEMORY SQLITE DATABASE
# ------------------------------------------------------------

con = sqlite3.connect(":memory:")

customers.to_sql("customers", con, index=False, if_exists="replace")
orders.to_sql("orders", con, index=False, if_exists="replace")


# ------------------------------------------------------------
# 6. SQL HELPER FUNCTION
# ------------------------------------------------------------

def sql(query):
    return pd.read_sql_query(query, con)


# ------------------------------------------------------------
# 7. CHECK THAT THE DATA HAS LOADED
# ------------------------------------------------------------

print("CUSTOMERS")
display(customers)

print("ORDERS")
display(orders)

print("A/B TEST DATA")
display(ab)

print("MARKETING CHANNEL DATA")
display(channels)

print("DATABASE SETUP COMPLETED SUCCESSFULLY.")

CUSTOMERS


,customer_id,city
0,101,Nairobi
1,102,nairobi
2,103,Mombasa
3,104,Kisumu
4,105,Nairobi
5,106,Nakuru
6,107,Mombasa
7,108,Nairobi
8,109,Eldoret
9,110,Kisumu


ORDERS


,order_id,customer_id,order_date,status,amount_kes
0,1001,101,2026-06-03,completed,2400.0
1,1002,102,2026-06-05,completed,1850.0
2,1003,103,2026-06-11,cancelled,3200.0
3,1004,104,2026-06-18,completed,2100.0
4,1005,101,2026-06-25,completed,2650.0
5,1006,105,2026-07-02,completed,1500.0
6,1006,105,2026-07-02,completed,1500.0
7,1007,106,2026-07-09,refunded,2900.0
8,1008,102,2026-07-14,completed,3100.0
9,1009,107,2026-07-21,completed,NaN


A/B TEST DATA


,version,visitors,conversions
0,Current,1000,50
1,New,1000,60


MARKETING CHANNEL DATA


,month,channel,visitors,conversions
0,September,Organic,40000,2400
1,September,Paid Social,10000,200
2,October,Organic,38000,2318
3,October,Paid Social,30000,630


DATABASE SETUP COMPLETED SUCCESSFULLY.


In [13]:
# ============================================================
# PART 1: SQL ANALYSIS
# S1: ORDER STATUS SUMMARY
# ============================================================

s1_result = sql("""
SELECT
    status,
    COUNT(DISTINCT order_id) AS number_of_orders
FROM orders
GROUP BY status
ORDER BY status;
""")

display(s1_result)

,status,number_of_orders
0,cancelled,2
1,completed,16
2,refunded,2


In [14]:
# ============================================================
# S2: COMPLETED REVENUE BY CITY
# ============================================================

s2_result = sql("""
WITH ranked_orders AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY order_id
               ORDER BY order_id
           ) AS rn
    FROM orders
)
SELECT
    TRIM(LOWER(c.city)) AS city,
    SUM(o.amount_kes) AS total_revenue_kes
FROM ranked_orders o
JOIN customers c
    ON o.customer_id = c.customer_id
WHERE
    o.rn = 1
    AND o.status = 'completed'
    AND o.amount_kes IS NOT NULL
GROUP BY TRIM(LOWER(c.city))
ORDER BY total_revenue_kes DESC;
""")

display(s2_result)

,city,total_revenue_kes
0,kisumu,187100.0
1,nairobi,20650.0
2,mombasa,5150.0
3,nakuru,3300.0


In [15]:
# ============================================================
# S3: CUSTOMERS WHO HAVE NEVER PLACED AN ORDER
# ============================================================

s3_result = sql("""
SELECT
    c.customer_id,
    c.city
FROM customers c
LEFT JOIN orders o
    ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL
ORDER BY c.customer_id;
""")

display(s3_result)

,customer_id,city
0,109,Eldoret
1,110,Kisumu


In [16]:
# ============================================================
# S4: REPEAT PURCHASE RATE
# ============================================================

s4_result = sql("""
WITH unique_orders AS (
    SELECT DISTINCT order_id, customer_id, status
    FROM orders
),
customer_orders AS (
    SELECT
        c.customer_id,
        COUNT(u.order_id) AS completed_orders
    FROM customers c
    LEFT JOIN unique_orders u
        ON c.customer_id = u.customer_id
        AND u.status = 'completed'
    GROUP BY c.customer_id
)
SELECT
    SUM(CASE WHEN completed_orders >= 2 THEN 1 ELSE 0 END)
        AS repeat_customers,
    COUNT(*) AS total_customers,
    ROUND(
        100.0 * SUM(
            CASE WHEN completed_orders >= 2 THEN 1 ELSE 0 END
        ) / COUNT(*), 2
    ) AS repeat_purchase_rate_percent
FROM customer_orders;
""")

display(s4_result)

,repeat_customers,total_customers,repeat_purchase_rate_percent
0,4,10,40.0


In [17]:
# ============================================================
# S5: MONTH-ON-MONTH COMPLETED REVENUE
# ============================================================

s5_result = sql("""
WITH unique_orders AS (
    SELECT
        order_id,
        order_date,
        status,
        amount_kes
    FROM orders
    GROUP BY order_id
),
monthly_revenue AS (
    SELECT
        strftime('%Y-%m', order_date) AS month,
        SUM(amount_kes) AS revenue_kes
    FROM unique_orders
    WHERE
        status = 'completed'
        AND amount_kes IS NOT NULL
    GROUP BY strftime('%Y-%m', order_date)
),
revenue_changes AS (
    SELECT
        month,
        revenue_kes,
        LAG(revenue_kes) OVER (ORDER BY month) AS previous_revenue
    FROM monthly_revenue
)
SELECT
    month,
    revenue_kes,
    CASE
        WHEN previous_revenue IS NULL THEN NULL
        ELSE ROUND(
            (revenue_kes - previous_revenue) * 100.0
            / previous_revenue, 2
        )
    END AS percentage_change
FROM revenue_changes
ORDER BY month;
""")

display(s5_result)

,month,revenue_kes,percentage_change
0,2026-06,9000.0,NaN
1,2026-07,6800.0,-24.44
2,2026-08,192000.0,2723.53
3,2026-09,10300.0,-94.64


## Part 3: Data Quality and Spreadsheets

### D1. Audit the Data

The following data quality issues concern me:

1. **Duplicate orders:** Order ID 1006 appears twice. I would investigate and remove the duplicate if confirmed.
2. **Missing values:** Order 1009 has a NULL amount. I would check the original transaction records.
3. **Unusual amount:** Order 1013 has KES 185,000, which may be an incorrect entry. I would verify it.
4. **Inconsistent city names:** Nairobi appears with different capitalisation and extra spaces. I would standardise the city names.
5. **Invalid customer reference:** Order 1020 belongs to customer ID 999, who is not in the customers table. I would investigate the missing customer record.
6. **Order statuses:** Cancelled and refunded orders should not be included in completed revenue calculations.

I would clean and validate the data before sharing final reports.

### D2. Spreadsheet Formulas

**1. Calculate total completed M-Pesa revenue:**

`=SUMIFS(E:E,D:D,"completed",F:F,"M-Pesa")`

**2. Retrieve each customer's city from the customers sheet:**

`=XLOOKUP(B2,customers!A:A,customers!B:B,"Not Found")`

The first formula adds the amounts for orders marked as completed and paid through M-Pesa. The second looks up a customer ID and returns the matching city from the customers sheet.

### D3. Mean or Median

I would use the median because the dataset contains an unusually high completed order of KES 185,000. This outlier can increase the mean and make it less representative of a typical order.

The median is less affected by extreme values and gives a better picture of the typical completed order value. Based on the deduplicated completed orders with non-missing amounts, the median is KES 2,450.

## Part 4: Statistics

### T1. Understanding P-values and Correlation

**a) What does a p-value tell us?**

A p-value helps us understand how likely it is to observe results as extreme as the ones obtained, assuming there is no real effect or difference. A small p-value, such as 0.05 or below, provides evidence against the assumption of no effect. However, it does not tell us the probability that our conclusion is correct.

**b) Does sending SMS cause customers to spend more?**

Not necessarily. Customers who receive SMS messages may already be more active or likely to spend. This shows an association, not necessarily causation. To establish causation, I would conduct a randomised experiment by dividing customers into treatment and control groups and comparing their spending.

In [18]:
ab["conversion_rate_percent"] = (
    ab["conversions"] / ab["visitors"] * 100
)

display(ab)

,version,visitors,conversions,conversion_rate_percent
0,Current,1000,50,5.0
1,New,1000,60,6.0


### T2. A/B Testing

The current version has a conversion rate of 5% (50 conversions out of 1,000 visitors), while the new version has a conversion rate of 6% (60 conversions out of 1,000 visitors).

The new version shows an improvement of 1 percentage point, or a 20% relative increase in conversion rate.

I would not recommend launching the new version immediately. I would first check whether the difference is statistically significant, ensure the test ran for a sufficient period, and confirm that the two groups were treated fairly. The results are promising, but more evidence is needed before making a final decision.

In [19]:
# Calculate conversion rate for each marketing channel
channels["conversion_rate_percent"] = (
    channels["conversions"] / channels["visitors"] * 100
)

display(channels)

# Calculate overall conversion rate for each month
monthly_summary = channels.groupby("month").agg(
    total_visitors=("visitors", "sum"),
    total_conversions=("conversions", "sum")
).reset_index()

monthly_summary["overall_conversion_rate_percent"] = (
    monthly_summary["total_conversions"] /
    monthly_summary["total_visitors"] * 100
)

display(monthly_summary)

,month,channel,visitors,conversions,conversion_rate_percent
0,September,Organic,40000,2400,6.0
1,September,Paid Social,10000,200,2.0
2,October,Organic,38000,2318,6.1
3,October,Paid Social,30000,630,2.1


,month,total_visitors,total_conversions,overall_conversion_rate_percent
0,October,68000,2948,4.335294
1,September,50000,2600,5.200000


## Part 5: Business Case – Falling Conversion Rate

### Analysis

September had 50,000 visitors and 2,600 conversions, giving an overall conversion rate of 5.20%.

October had 68,000 visitors and 2,948 conversions, giving an overall conversion rate of approximately 4.34%.

Although total conversions increased, the overall conversion rate decreased because paid social traffic grew significantly as a share of total visitors. Paid social accounted for 20% of visitors in September and approximately 44.12% in October.

However, conversion rates within both channels improved slightly:
- Organic: 6.0% in September to 6.1% in October.
- Paid Social: 2.0% in September to 2.1% in October.

### Recommendation

I would not recommend pausing paid social immediately because its conversion rate improved slightly from 2% to 2.1%, while organic conversion also improved. I recommend investigating the quality and source of the increased paid traffic and optimising the campaigns before making a decision.

## Part 6: Questions for the Employer

1. What would the main responsibilities and expectations be for someone joining this role as a Junior Data Analyst?

2. Which tools and technologies does the data team mainly use for data analysis and reporting?

3. Are there opportunities for learning, mentorship and career growth within the team?

4. What would success look like for someone in this position during their first three months?